# 05h - HayFlow-Hines representation and raw-scale forensics

This notebook follows the 05g train-fit no-go. It reuses the exact registered 12-pair support, audits all feature surfaces before clipping, localizes the held-out input excursion, measures the irreducible train-only linear projection residual, and compares three small bounded nonlinear controls on train and development only. The frozen H2 checkpoint is evaluated on held-out inputs solely to extract diagnostic features; held-out boundary-voltage targets are never loaded, candidate heads are never evaluated there, and no rollout or full training is present.

## 1. Coherent checkout and GPU runtime

In [ ]:
import os, subprocess, sys
from pathlib import Path
WORKSPACE = Path('/kaggle/working/hayflow_workspace')
ELM_REPO = WORKSPACE / 'elmneuron'
if not ELM_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Zagred47/giada.git', str(ELM_REPO)], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pandas', 'pyarrow', 'pyyaml', 'matplotlib'], check=True)
sys.path.insert(0, str(ELM_REPO))
REVISION = subprocess.check_output(['git', '-C', str(ELM_REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Revision:', REVISION)

In [ ]:
import h5py, json, numpy as np, pandas as pd, pyarrow, torch, yaml
assert torch.cuda.is_available(), 'Attiva una GPU Kaggle prima di eseguire 05h.'
print({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0), 'experiment': 'representation and raw-scale forensics'})

## 2. Immutable inputs
Sono richiesti il composite targeted, il dataset base e gli artefatti esatti da 05b a 05g. La directory estratta da Kaggle e lo ZIP originale sono entrambi accettati; tutti i membri critici di 05g sono verificati tramite SHA-256.

In [ ]:
import shutil, zipfile
INPUT_ROOT = Path('/kaggle/input')
def extract_zip_safely(source, destination):
    source, destination = Path(source), Path(destination)
    marker = destination / '.source_size'; stamp = str(source.stat().st_size)
    if marker.is_file() and marker.read_text().strip() == stamp: return destination
    if destination.exists(): shutil.rmtree(destination)
    destination.mkdir(parents=True); root = destination.resolve()
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            assert target == root or root in target.parents, member.filename
        archive.extractall(destination)
    marker.write_text(stamp); return destination
def first_existing(candidates, message):
    found = next((Path(p).resolve() for p in candidates if Path(p).exists()), None)
    assert found is not None, message
    return found
def artifact_source(env_name, zip_name, marker, message):
    candidates = [Path(os.environ[env_name]).expanduser()] if os.environ.get(env_name) else []
    candidates += list(INPUT_ROOT.rglob(zip_name))
    candidates += [p.parent for p in INPUT_ROOT.rglob(marker)]
    return first_existing(candidates, message)

topup_candidates = [Path(os.environ['HAYFLOW_TOPUP_V3']).expanduser()] if os.environ.get('HAYFLOW_TOPUP_V3') else []
topup_candidates += list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))
topup_candidates += [p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE = first_existing(topup_candidates, 'Top-up BAP v3 non trovato.')
TOPUP_ROOT = extract_zip_safely(TOPUP_SOURCE, '/kaggle/working/hayflow05h_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates = list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'))
assert len(manifest_candidates) == 1, manifest_candidates
COMPOSITE_MANIFEST = manifest_candidates[0]
base_candidates = [Path(os.environ['HAYFLOW_BASE_DATASET']).expanduser()] if os.environ.get('HAYFLOW_BASE_DATASET') else []
base_candidates += [p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]
base_candidates += [p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE = first_existing(base_candidates, 'Dataset base targeted v1.1 non trovato.')
CHECKPOINT_05B_SOURCE = artifact_source('HAYFLOW_05B_ARTIFACT', 'hayflow_hines_canary_v2.zip', 'canary_models.pt', 'Artefatto 05b non trovato.')
if CHECKPOINT_05B_SOURCE.name == 'checkpoints': CHECKPOINT_05B_SOURCE = CHECKPOINT_05B_SOURCE.parent
ARTIFACT_05C_SOURCE = artifact_source('HAYFLOW_05C_ARTIFACT', 'hayflow_hines_causal_isolation.zip', 'checkpoint_forensics.json', 'Artefatto 05c non trovato.')
ARTIFACT_05D_SOURCE = artifact_source('HAYFLOW_05D_ARTIFACT', 'hayflow_hines_residual_conditioning.zip', 'free_residual_report.json', 'Artefatto 05d non trovato.')
ARTIFACT_05E_SOURCE = artifact_source('HAYFLOW_05E_ARTIFACT', 'hayflow_hines_segment_capacity.zip', 'capacity_probe_report.json', 'Artefatto 05e non trovato.')
ARTIFACT_05F_SOURCE = artifact_source('HAYFLOW_05F_ARTIFACT', 'hayflow_hines_segment_micro_canary.zip', 'micro_canary_report.json', 'Artefatto 05f non trovato.')
ARTIFACT_05G_SOURCE = artifact_source('HAYFLOW_05G_ARTIFACT', 'hayflow_hines_optimization_audit.zip', 'optimization_support.json', 'Artefatto 05g non trovato.')
print({'manifest': str(COMPOSITE_MANIFEST), 'base': str(BASE_SOURCE), '05b': str(CHECKPOINT_05B_SOURCE), '05c': str(ARTIFACT_05C_SOURCE), '05d': str(ARTIFACT_05D_SOURCE), '05e': str(ARTIFACT_05E_SOURCE), '05f': str(ARTIFACT_05F_SOURCE), '05g': str(ARTIFACT_05G_SOURCE)})

## 3. Composite and cryptographic provenance preflight

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started, hash_last = {}, {}
def hash_progress(name, done, total):
    now = time.monotonic(); hash_started.setdefault(name, now); percent = int(100 * done / total)
    if percent >= hash_last.get(name, -5) + 5 or done == total:
        elapsed = now - hash_started[name]; rate = done / max(elapsed, 1e-9); eta = (total - done) / max(rate, 1e-9)
        print(f'[HayFlow 05h][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min', flush=True); hash_last[name] = percent
bundle = prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST, base_source=BASE_SOURCE, progress=hash_progress)
display({'valid': bundle.manifest['valid'], 'fingerprint': bundle.fingerprint, 'transition_count': bundle.transition_count, 'physical_merge': bundle.manifest['physical_merge_performed']})
assert bundle.manifest['valid'] and bundle.transition_count == 29880 and not bundle.manifest['physical_merge_performed']

In [ ]:
from src.hayflow_model import HinesCapacityConfig, HinesConditioningConfig, HinesIsolationConfig, HinesOptimizationAuditConfig, HinesPrototypeExperimentConfig, HinesRepresentationForensics, HinesRepresentationForensicsConfig, HinesSegmentCanaryConfig
base_config = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_optimization_audit.yml').read_text())
forensic_config = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_representation_forensics.yml').read_text())
model_config = HinesPrototypeExperimentConfig.from_mapping(base_config['model_experiment'])
isolation_config = HinesIsolationConfig.from_mapping(base_config['isolation'])
conditioning_config = HinesConditioningConfig.from_mapping(base_config['conditioning'])
capacity_config = HinesCapacityConfig.from_mapping(base_config['capacity'])
canary_config = HinesSegmentCanaryConfig.from_mapping(base_config['micro_canary'])
audit_config = HinesOptimizationAuditConfig.from_mapping(base_config['optimization_audit'])
representation_config = HinesRepresentationForensicsConfig.from_mapping(forensic_config['representation_forensics'])
OUTPUT_DIR = Path('/kaggle/working/artifacts/hayflow_hines_representation_forensics')
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
session = HinesRepresentationForensics(bundle, OUTPUT_DIR, model_config, isolation_config, conditioning_config, capacity_config, canary_config, audit_config, representation_config, CHECKPOINT_05B_SOURCE, ARTIFACT_05C_SOURCE, ARTIFACT_05D_SOURCE, ARTIFACT_05E_SOURCE, ARTIFACT_05F_SOURCE, ARTIFACT_05G_SOURCE, code_revision=REVISION)
prepare_report = session.prepare_forensics()
display({'revision': REVISION, 'support_sha256': prepare_report['support_sha256'], '05g': prepare_report['artifact_05g'], 'full_training_authorized': prepare_report['full_training_authorized']})
assert not prepare_report['full_training_authorized']

## 4. Pre-clipping raw-scale and zero-causal forensics
Si misurano H2, H2 con input causali azzerati e input causali diretti prima di qualunque clipping. I target held-out non vengono caricati. Questa cella localizza campione, segmento e feature responsabili delle escursioni.

In [ ]:
raw_report = session.run_raw_scale_forensics()
display({k: raw_report[k] for k in ['valid', 'heldout_boundary_targets_materialized', 'heldout_event_targets_materialized', 'raw_heldout_ood_blocker', 'inferred_anomaly_origin', 'h2_raw_heldout_to_train_max_norm_ratio', 'causal_raw_heldout_to_train_max_norm_ratio', 'zero_causal_h2_heldout_to_train_max_norm_ratio', 'normalized_teacher_state_heldout_to_train_max_ratio']})
display(raw_report['state_and_voltage_surfaces'])
display(pd.DataFrame(raw_report['top_outliers']).head(30))
assert not raw_report['heldout_boundary_targets_materialized']
assert not raw_report['heldout_event_targets_materialized']

## 5. Train-only linear projection forensics

In [ ]:
projection_report = session.run_projection_forensics()
display({k: projection_report[k] for k in ['valid', 'minimum_design_rank', 'maximum_design_rank', 'rank_error_correlation', 'segment_count_with_projection_rmse_above_1mv', 'pair_metrics']})
display(pd.DataFrame(projection_report['worst_segments']))
assert not projection_report['heldout_targets_used']

## 6. Bounded nonlinear representation controls
Tre piccoli head condivisi confrontano feature H2, input causali diretti e H2+causali. H2 resta congelato; il residuo è limitato a ±120 mV. Checkpoint ed early stopping usano soltanto train e la coppia development registrata. La cella stampa avanzamento ed ETA.

In [ ]:
controls_report = session.run_bounded_representation_controls()
display(pd.DataFrame(controls_report['family_summary']))
assert controls_report['valid']
assert not controls_report['heldout_boundary_targets_materialized']
assert not controls_report['heldout_event_targets_materialized']
assert controls_report['heldout_frozen_h2_feature_extraction_performed']
assert not controls_report['heldout_candidate_head_inference_performed']

## 7. Final scoped decision

In [ ]:
final_report = session.finalize_forensics(raw_report, projection_report, controls_report)
display({'valid': final_report['valid'], 'decision': final_report['decision'], 'diagnosis': final_report['diagnosis'], 'heldout_contract': final_report['heldout_contract'], 'next_step': final_report['next_step']})
assert final_report['valid']
assert not final_report['full_training_authorized']
assert not final_report['heldout_contract']['boundary_targets_materialized']
assert not final_report['heldout_contract']['event_targets_materialized']
assert final_report['heldout_contract']['frozen_h2_feature_extraction_performed']
assert not final_report['heldout_contract']['candidate_head_inference_performed']
from IPython.display import Image, display
display(Image(filename=str(OUTPUT_DIR / 'representation_forensics.png')))

## 8. Create and download the diagnostic ZIP

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript, display
zip_base = Path('/kaggle/working/hayflow_hines_representation_forensics')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
payload = base64.b64encode(zip_path.read_bytes()).decode('ascii'); filename = zip_path.name
display(Javascript(f"""
const binary = atob('{payload}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url; anchor.download = '{filename}';
document.body.appendChild(anchor); anchor.click(); anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
"""))
print({'zip': str(zip_path), 'size_mib': round(zip_path.stat().st_size / 2**20, 2), 'download': 'avviato dal browser'})